In [22]:
!uv pip install langchain-core==0.2.38
!uv pip install langchain-google-genai==1.0.2
# pip install langchain-core==0.1.52
! uv pip install langchain==0.1.17
# pip install langchain-google-genai==1.0.2
! uv pip install ragas==0.1.16
! uv pip install google-generativeai python-dotenv

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Checked 1 package in 3ms
Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 44 packages in 224ms                                        
⠙ Preparing packages... (0/7)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/51.77 KiB           
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/51.77 KiB         
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/51.77 KiB         
grpcio-status        ------------------------------     0 B/14.11 KiB
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/51.77 KiB         
grpcio-status        ------------------------------     0 B/14.11 KiB
⠙ Preparing packages... (0/7)30m------------ 32.00 KiB/51.77 KiB         
grpcio-status        -----------------------------

In [12]:
! uv pip install -U langchain langchain-core langchain-google-genai ragas qdrant-client langsmith python-dotenv

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 106 packages in 219ms                                       
⠙ Preparing packages... (0/50)                                                  
⠙ Preparing packages... (0/50)------------------     0 B/119.69 KiB          
⠙ Preparing packages... (0/50)------------------     0 B/119.69 KiB          
dill                 ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/50)------------------     0 B/119.69 KiB          
dill                 ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/50)------------------     0 B/119.69 KiB          
dill                 ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/50)------------------     0 B/119.69 KiB          
click                ------------------------------     0 B/116.45 KiB
dill                 ------------------------------

In [ ]:
import openai
from qdrant_client import QdrantClient

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy

2026-07-08 20:28:49.531040005 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


### Download an example reference data point from LangSmith

In [2]:
client = Client()

In [3]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [4]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('3d0f1c4c-10e6-44e2-9891-5da8a43515c8'), created_at=datetime.datetime(2026, 6, 27, 2, 8, 44, 937704, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 6, 27, 2, 8, 44, 937704, tzinfo=datetime.timezone.utc), example_count=30, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None)

In [5]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

{'ground_truth': "I'm sorry, but the specific duration of the warranty for the Garmin 890 RV GPS Navigator is not mentioned in the product description, only that it comes with a 'Limited Warranty'.",
 'reference_context_ids': [],
 'reference_descriptions': []}

In [6]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs

{'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?'}

In [7]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

### RAG Pipeline

In [8]:
# def get_embedding(text, model="text-embedding-3-small"):
#     response = openai.embeddings.create(
#         input=text,
#         model=model,
#     )

#     return response.data[0].embedding


# Load API key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

# Initialize client
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)


# @traceable(name="embed_query", run_type="embedding", metadata={"ls_provider": "google", "ls_model_name": "gemini-embedding-001"})
# , run_type="embedding", metadata={"ls_provider": "google", "ls_model_name": "gemini-embedding-001"})
# 3. Define the embedding function with custom dimensionality
def get_embedding(text, model="gemini-embedding-001",):
    """
    Generates an embedding vector for the given text using Gemini's current model.
    """
    response = gemini_client.models.embed_content(
        model="gemini-embedding-001", 
        contents=text,
        config=types.EmbedContentConfig(
        #     task_type=task_type
        #     # 🔴 THE FIX: Force the output to be exactly 1536 dimensions
        #     # output_dimensionality=1536  
        )
    )
    # current_run = get_current_run_tree()
    # if current_run:
    #     current_run.metadata["usage_metadata"] = {
    #         "input_tokens": response.usage.prompt_tokens,
    #         "total_tokens": response.usage.total_tokens,
    #     }
    # Note: EmbedContentResponse does not contain a 'usage' attribute in the 
    # current version of the google-genai SDK, so we skip setting token usage for embeddings.
    # Time and duration are still tracked by LangSmith automatically.
    
    return response.embeddings[0].values





def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


# def generate_answer(prompt):

#     response = openai.chat.completions.create(
#         model="gpt-5-nano",
#         messages=[{"role": "system", "content": prompt}],
#         reasoning_effort="minimal"
#     )

#     return response.choices[0].message.content


def generate_answer(prompt, model_name="gemini-2.5-flash"):
    """
    Generates a response using the specified Gemini model.
    """
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
        # You can add config here if you need system instructions or temperature control
    )
    
    return response.text


def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [9]:
rag_pipeline("Can I get some charger?", top_k=5)

/tmp/ipykernel_3008004/1365509021.py:138: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


{'answer': "We have several charging cables available:\n\n*   **Mixblu Charger Cable Replacement for Fitbit Inspire 3:** This is a 2-pack of 3.3ft replacement charging cables specifically designed for the Fitbit Inspire 3. They offer fast and stable charging and are made of durable ABS plastic and TPE.\n*   **5 in 1 USB C to Multi Charging Cable (3M/10Ft):** This is a versatile cable with USB A/USB C to Lightning, Type C, and Micro USB connectors. It can charge three devices simultaneously and is compatible with various iPhones (14 series down to 7), Samsung Galaxy S10/Note 8, Nexus, LG, Pixel, HTC, OnePlus, Huawei, XiaoMi, Samsung Galaxy S7/S6/S4, Kindle, Bluetooth headsets, external batteries, and PS4 controllers. It is Apple MFi Certified and 10ft long. Note that it's for charging only, not data sync, and the USB C port does not work with iPads or other high-power requirement devices.\n*   **iPhone Charger Cord Lightning Cables (3Pack 3ft):** These are Apple MFi Certified USB A char

### RAGAS metrics

In [10]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

ModuleNotFoundError: No module named 'ragas.dataset_schema'

In [12]:
! uv pip install langchain_google_genai

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 38 packages in 166ms                                        
⠙ Preparing packages... (0/6)                                                   
⠙ Preparing packages... (0/6)-------------------     0 B/250.76 KiB          
⠙ Preparing packages... (0/6)------------------- 16.00 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)------------------- 32.00 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)------------------- 48.00 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)------------------- 61.91 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)------------------- 77.91 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)------------------- 93.91 KiB/250.76 KiB        
⠙ Preparing packages... (0/6)2m----------------- 109.91 KiB/250.76 KiB       
⠙ Preparing packages... (0/6)2m----------------- 109.91 KiB/250.76 KiB       
filetype             --

In [14]:
! uv pip install --upgrade langchain langchain-core langchain-google-genai

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 45 packages in 152ms                                        
⠙ Preparing packages... (0/26)                                                  
⠙ Preparing packages... (0/26)------------------     0 B/219.00 KiB          
⠙ Preparing packages... (0/26)------------------     0 B/219.00 KiB          
anyio                ------------------------------     0 B/121.95 KiB
⠙ Preparing packages... (0/26)------------------     0 B/219.00 KiB          
anyio                ------------------------------     0 B/121.95 KiB
⠙ Preparing packages... (0/26)------------------     0 B/219.00 KiB          
anyio                ------------------------------     0 B/121.95 KiB
⠙ Preparing packages... (0/26)------------------     0 B/219.00 KiB          
pyasn1               ------------------------------     0 B/82.43 KiB
anyio                ------------------------------ 

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# --- Available Gemini LLM Models ---
# "gemini-1.5-flash" : Fast, cost-efficient, great for Ragas evaluations
# "gemini-1.5-pro"   : High reasoning capability, better for complex context analysis
# "gemini-2.0-flash" : Higher performance, latest generation (if available in your region)
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-1.5-flash")
)

# --- Available Gemini Embedding Models ---
# "models/text-embedding-004" : Newer, high-performance model (recommended)
# "models/embedding-001"      : Standard, legacy support model
ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/langchain_core/exceptions.py)

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# --- Available Gemini LLM Models ---
# "gemini-1.5-flash" : Fast, cost-efficient, great for Ragas evaluations
# "gemini-1.5-pro"   : High reasoning capability, better for complex context analysis
# "gemini-2.0-flash" : Higher performance, latest generation (if available in your region)
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-1.5-flash")
)

# --- Available Gemini Embedding Models ---
# "models/text-embedding-004" : Newer, high-performance model (recommended)
# "models/embedding-001"      : Standard, legacy support model
ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/langchain_core/exceptions.py)

In [17]:
! uv pip show langchain-core langchain-google-genai

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Name: langchain-core
Version: 1.4.9
Location: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-google-genai, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, ragas
---
Name: langchain-google-genai
Version: 4.2.7
Location: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages
Requires: filetype, google-genai, langchain-core, pydantic
Required-by:


In [18]:
! uv pip install -U langchain-core

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 30 packages in 97ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/180.58 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)------------------- 60.47 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)m------------------ 76.47 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)--------------- 92.47 KiB/180.58 KiB        
⠙ Preparing packages... (0/1)30m------------ 108.47 KiB/180.58 KiB       
⠙ Preparing packages... (0/1)---------- 124.47 KiB/180.58 KiB       
⠙ Preparing packages... (0/1)---------- 

In [19]:
! uv sync

Resolved 182 packages in 1ms
Checked 170 packages in 1ms


In [2]:
!uv pip install langchain-core==0.2.38
!uvpip install langchain-google-genai==1.0.2

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 23 packages in 69ms                                         
⠙ Preparing packages... (0/4)                                                   
⠙ Preparing packages... (0/4)-------------------     0 B/27.50 KiB           
⠙ Preparing packages... (0/4)0m------------- 16.00 KiB/27.50 KiB         
⠙ Preparing packages... (0/4)---------- 27.50 KiB/27.50 KiB         
⠙ Preparing packages... (0/4)0/4)                                                        
⠙ Preparing packages... (0/4)-------------------     0 B/387.15 KiB          
⠙ Preparing packages... (0/4)------------------- 14.83 KiB/387.15 KiB        
⠙ Preparing packages... (0/4)------------------- 30.83 KiB/387.15 KiB        
⠙ Preparing packages... (0/4)------------------- 46.83 KiB/387.15 KiB        
⠙ Preparing packages... (0/4)------------------- 46.83 KiB/387.15 KiB        
langsmith            ---

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# --- Available Gemini LLM Models ---
# "gemini-1.5-flash" : Fast, cost-efficient, great for Ragas evaluations
# "gemini-1.5-pro"   : High reasoning capability, better for complex context analysis
# "gemini-2.0-flash" : Higher performance, latest generation (if available in your region)
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-1.5-flash")
)

# --- Available Gemini Embedding Models ---
# "models/text-embedding-004" : Newer, high-performance model (recommended)
# "models/embedding-001"      : Standard, legacy support model
ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/langchain_core/exceptions.py)

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# --- Available Gemini LLM Models ---
# "gemini-1.5-flash" : Fast, cost-efficient, great for Ragas evaluations
# "gemini-1.5-pro"   : High reasoning capability, better for complex context analysis
# "gemini-2.0-flash" : Higher performance, latest generation (if available in your region)
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-1.5-flash")
)

# --- Available Gemini Embedding Models ---
# "models/text-embedding-004" : Newer, high-performance model (recommended)
# "models/embedding-001"      : Standard, legacy support model
ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

/tmp/ipykernel_2920802/1989966782.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/tmp/ipykernel_2920802/1989966782.py:16: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [16]:
reference_input

{'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?'}

In [17]:
reference_output

{'ground_truth': "I'm sorry, but the specific duration of the warranty for the Garmin 890 RV GPS Navigator is not mentioned in the product description, only that it comes with a 'Limited Warranty'.",
 'reference_context_ids': [],
 'reference_descriptions': []}

In [18]:
result = rag_pipeline(reference_input["question"])

/tmp/ipykernel_2920802/1365509021.py:138: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


In [19]:
result

{'answer': 'Based on the available products, the Garmin 890 8-inch RV GPS Navigator Bundle comes with a Limited Warranty. The exact duration of this limited warranty is not specified.',
 'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?',
 'retrieved_context_ids': ['B08BX2L8F2',
  'B0B3MMP22L',
  'B0BLV7YH2W',
  'B0BB1YKXL1',
  'B09QGNB537'],
 'retrieved_context': ['Garmin 890 8-inch RV GPS Navigator Bundle with Car Charger Expander and Hard Shell EVA Case for Tablets/GPS (010-02425-00) Built in Wi-Fi connectivity makes updating your maps of North America a breeze. This large 8" GPS navigator features a bright, high-resolution edge-to-edge touchscreen display so you can easily see important information Built-in Wi-Fi connectivity makes it easy to keep your maps and software up to date without using a computer IN THE BOX: RV 890 | Vehicle suction cup with powered magnetic mount | Screw down mount | 1" ball adapter with AMPS plate | Vehicle power cable | USB c

In [23]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [24]:
await ragas_faithfulness(result, "")

ImportError: cannot import name 'TracerSessionV1' from 'langchain_core.tracers.schemas' (/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/langchain_core/tracers/schemas.py)

In [ ]:
async def ragas_responce_relevancy(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_responce_relevancy(result, "")

In [ ]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
            retrieved_context_ids=run["retrieved_context_ids"],
            reference_context_ids=example["reference_context_ids"]
        )
    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_precision_id_based(result, reference_output)

In [ ]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
            retrieved_context_ids=run["retrieved_context_ids"],
            reference_context_ids=example["reference_context_ids"]
        )
    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)

In [13]:
import os
import asyncio
from dotenv import load_dotenv

# Google & Qdrant Imports
from google import genai
from google.genai import types
from qdrant_client import QdrantClient

# LangSmith Imports
from langsmith import Client

# Modern LangChain & Ragas Imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas import SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

# ---------------------------------------------------------
# 1. Environment & Client Setup
# ---------------------------------------------------------
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
ls_client = Client() # LangSmith Client

# ---------------------------------------------------------
# 2. Initialize Modern RAGAS LLMs & Embeddings (THE FIX)
# ---------------------------------------------------------
# Instead of deprecated wrappers, we pass Langchain models into the Ragas factories.
base_gemini_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
ragas_llm = llm_factory(model=base_gemini_llm)

base_gemini_emb = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
ragas_embeddings = embedding_factory(model=base_gemini_emb)

# ---------------------------------------------------------
# 3. Core RAG Pipeline Components
# ---------------------------------------------------------
def get_embedding(text, model="gemini-embedding-001"):
    response = gemini_client.models.embed_content(
        model=model, 
        contents=text
    )
    return response.embeddings[0].values

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question):
    return f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

def generate_answer(prompt, model_name="gemini-1.5-flash"): # Updated to 1.5-flash as 2.5 is not universally standard yet
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return response.text

def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

# ---------------------------------------------------------
# 4. Fetch LangSmith Example
# ---------------------------------------------------------
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=10))

reference_input = examples[0].inputs
reference_output = examples[0].outputs

# Run the pipeline to get the result to evaluate
print("Running RAG Pipeline...")
result = rag_pipeline(reference_input["question"])
print("Generated Answer:", result["answer"])

# ---------------------------------------------------------
# 5. RAGAS Evaluation Functions (Async)
# ---------------------------------------------------------
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    # Relevancy requires both LLM and Embeddings
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

# ---------------------------------------------------------
# 6. Execute Evaluations
# ---------------------------------------------------------
async def run_evaluations():
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(result, reference_output)
    print(f"Faithfulness Score: {faithfulness_score}")

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(result, reference_output)
    print(f"Response Relevancy Score: {relevancy_score}")

# Run the async evaluation block natively in Jupyter
await run_evaluations()

ImportError: cannot import name 'SingleTurnSample' from 'ragas' (/home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv/lib/python3.12/site-packages/ragas/__init__.py)